In [8]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType, IntegerType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *
import time


spark = (
    SparkSession.builder
    .appName("Column pruning")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext


In [9]:
guitarsDF = (
    spark.read.json("src/main/resources/data/guitars/guitars.json")
)

guitarsDF.show(5)

+--------------------+---+------+------------+
|          guitarType| id|  make|       model|
+--------------------+---+------+------------+
|Electric double-n...|  0|Gibson|    EDS-1275|
|            Electric|  5|Fender|Stratocaster|
|            Electric|  1|Gibson|          SG|
|            Acoustic|  2|Taylor|         914|
|            Electric|  3|   ESP|        M-II|
+--------------------+---+------+------------+



In [10]:
bandsDF = (
    spark.read.json("src/main/resources/data/bands/bands.json")
)

bandsDF.show(5)

+-----------+---+------------+----+
|   hometown| id|        name|year|
+-----------+---+------------+----+
|     Sydney|  1|       AC/DC|1973|
|     London|  0|Led Zeppelin|1968|
|Los Angeles|  3|   Metallica|1981|
|  Liverpool|  4| The Beatles|1960|
+-----------+---+------------+----+



In [11]:
guitarPlayersDF = (
    spark.read.json("src/main/resources/data/guitarPlayers/guitarPlayers.json")
)

guitarPlayersDF.show(5)


+----+-------+---+------------+
|band|guitars| id|        name|
+----+-------+---+------------+
|   0|    [0]|  0|  Jimmy Page|
|   1|    [1]|  1| Angus Young|
|   2| [1, 5]|  2|Eric Clapton|
|   3|    [3]|  3|Kirk Hammett|
+----+-------+---+------------+



In [12]:
joinCondition = guitarPlayersDF["band"] == bandsDF["id"]

In [13]:
guitaristsBandsDF = guitarPlayersDF.join(bandsDF, joinCondition, "inner")
guitaristsBandsDF.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [band#222L], [id#185L], Inner, BuildLeft, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=153]
   :  +- Filter isnotnull(band#222L)
   :     +- FileScan json [band#222L,guitars#223,id#224L,name#225] Batched: false, DataFilters: [isnotnull(band#222L)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/workspace/src/main/resources/data/guitarPlayers/guitarPlayers.json], PartitionFilters: [], PushedFilters: [IsNotNull(band)], ReadSchema: struct<band:bigint,guitars:array<bigint>,id:bigint,name:string>
   +- Filter isnotnull(id#185L)
      +- FileScan json [hometown#184,id#185L,name#186,year#187L] Batched: false, DataFilters: [isnotnull(id#185L)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/workspace/src/main/resources/data/bands/bands.json], PartitionFilters: [], PushedFilters: [IsNotNull(id)], ReadSchema: struct<hometown:string,id:bigint

In [14]:
guitaristsWithoutBandsDF = guitarPlayersDF.join(bandsDF, joinCondition, "left_anti")

guitaristsWithoutBandsDF.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [band#222L], [id#185L], LeftAnti, BuildRight, false
   :- FileScan json [band#222L,guitars#223,id#224L,name#225] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/workspace/src/main/resources/data/guitarPlayers/guitarPlayers.json], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<band:bigint,guitars:array<bigint>,id:bigint,name:string>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=172]
      +- Filter isnotnull(id#185L)
         +- FileScan json [id#185L] Batched: false, DataFilters: [isnotnull(id#185L)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/workspace/src/main/resources/data/bands/bands.json], PartitionFilters: [], PushedFilters: [IsNotNull(id)], ReadSchema: struct<id:bigint>




26/04/07 01:22:08 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
# notice that spark only selects the column relevant for the join id#185L 
# column pruning = cutoff columns that are not relevant
# = shrinks DF
# spark can do this right off the bat before joins

# this is particular useful for joins and groups

Notice below that when we only selected after joins we can see that spark already only the columns that we care about aka band id and name and id and name.

Then the ids get dropped in the last step
Spark tends to drop columns as early as possible, this should be YOUR goal as well.

When spark does not do this you should do column pruning

In [16]:
# project and filter push down
namesDF = guitaristsBandsDF.select(guitarPlayersDF["name"], bandsDF["name"])

namesDF.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [name#225, name#186]
   +- BroadcastHashJoin [band#222L], [id#185L], Inner, BuildRight, false
      :- Filter isnotnull(band#222L)
      :  +- FileScan json [band#222L,name#225] Batched: false, DataFilters: [isnotnull(band#222L)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/workspace/src/main/resources/data/guitarPlayers/guitarPlayers.json], PartitionFilters: [], PushedFilters: [IsNotNull(band)], ReadSchema: struct<band:bigint,name:string>
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=200]
         +- Filter isnotnull(id#185L)
            +- FileScan json [id#185L,name#186] Batched: false, DataFilters: [isnotnull(id#185L)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/workspace/src/main/resources/data/bands/bands.json], PartitionFilters: [], PushedFilters: [IsNotNull(id)], ReadSchema: struct<id:bigint,name:string>




Notice below:

We can see that spark does column pruning like
FileScan json [id#147L,make#148]
FileScan json [id#185L,name#186]
FileScan json [band#222L,guitars#223,name#225]

spark only includes the MINIMUM amount of cols needed


Notice that the final project with the upper function is being done LAST

## LESSON:
If you anticpicate that the joined dataframe after the join will be much larger than the table on whose column you are applying the map-side operation ed "* 5" or "upper", do this operation on the small table FIRST. Particularly useful for outer joins

On the contrary if the joined table will be much SMALLER then do the map onb the joined table.

TLDR; when joining and doing maps, do the map on the smaller table either 
1. the table before the join if its smaller 
2. or the table after the join (the joined table) if that table is smaller.

In [18]:
rockDF = (
    guitarPlayersDF
    .join(bandsDF, joinCondition)
    .join(guitarsDF, array_contains(guitarPlayersDF["guitars"], guitarsDF["id"]))
)

essentialsDF = rockDF.select(guitarPlayersDF["name"], bandsDF["name"], upper(guitarsDF["make"]))

essentialsDF.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [name#225, name#186, upper(make#148) AS upper(make)#334]
   +- BroadcastNestedLoopJoin BuildRight, Inner, array_contains(guitars#223, id#147L)
      :- Project [guitars#223, name#225, name#186]
      :  +- BroadcastHashJoin [band#222L], [id#185L], Inner, BuildRight, false
      :     :- Filter (isnotnull(band#222L) AND isnotnull(guitars#223))
      :     :  +- FileScan json [band#222L,guitars#223,name#225] Batched: false, DataFilters: [isnotnull(band#222L), isnotnull(guitars#223)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/workspace/src/main/resources/data/guitarPlayers/guitarPlayers.json], PartitionFilters: [], PushedFilters: [IsNotNull(band), IsNotNull(guitars)], ReadSchema: struct<band:bigint,guitars:array<bigint>,name:string>
      :     +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=247]
      :        +- Filter isnotnull(id#185L)
      :           +- 